# Sanity checks: filtered European panel

Checks the filtered European data (`22_filter_prices_eu.py` → `40_build_monthly_universe.py --region EU`)
against outside benchmarks and for internal consistency:

1. **Universe** per country over time (stocks, market cap).
2. **Fama-French Europe market.** French's Europe region covers 16 countries (AT BE DK FI FR DE GR IE IT NL NO PT ES
   SE CH UK), returns in **USD**, RF = one-month U.S. T-bill. We rebuild it as the value-weighted (lagged EUR market
   cap) return of these 16 countries, convert EUR → USD at month-end rates, and compare with `Mkt-RF + RF`.
   Section 2b tests the currency assumption empirically (USD vs. EUR version of our series).
3. **Datastream Total Market indices per country** (`TOTMK..`, `RI~E`, monthly): country VW returns vs. the
   index (needs the index download; skipped until the file exists).
4. **Local vs. EUR returns**: identical after euro adoption, different only for non-euro quotes.
5. Optional: **Top 600** by market cap (STOXX 600-like, no free-float weights) vs. a STOXX 600 / MSCI Europe series.

**Expectations (VW, monthly):** correlation with FF Europe ≳ 0.98, beta ≈ 1, alpha not significant, tracking
error of a few percent per year (FF uses Datastream/Worldscope too but other filters and a different universe
before 1990s coverage expands). Per-country correlations with the Datastream indices ≳ 0.95 for large markets;
small markets (CY, MT, Baltics, BG, RO) are noisier.

Functions: `src/datastream/eu_evaluation.py` (European helpers) and `src/datastream/evaluation.py` (comparison).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from datastream import eu_evaluation as eu
from datastream import evaluation as ev
from datastream.ff_data import get_ff_region_factors

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.width", 160)

# ── configuration ───────────────────────────────────────────────────────────────────────────────────
DATA_PATH = Path("D:/Datastream/PriceData/EU/processed")
PENNY_PERCENTILE = 0.25
UNIVERSE_FILE = DATA_PATH / f"monthly_universe_{PENNY_PERCENTILE}.parquet"
FX_FILE = Path("D:/Datastream/Benchmarks/DEXUSEU.csv")        # FRED daily USD per EUR (EXUSEU = monthly averages)
TOTMK_FILE = Path("D:/Datastream/Benchmarks/EU_TOTMK_RI_E.xlsx")  # Datastream Total Market indices, RI~E, monthly
STOXX_FILE = Path("D:/Datastream/Benchmarks/STOXX600_RI_E.xlsx")  # optional
COUNTRIES_FILE = Path("../config/eu_countries.csv")
START = "1999-01"                                              # analysis start (download starts 1992-12)
HAC_LAGS = 6

## 1. Data and universe

In [ ]:
uni = pd.read_parquet(UNIVERSE_FILE)
m = eu.monthly_returns(uni)
m = m[m["Month"] >= pd.Period(START, "M")]
print(f"{len(m):,} stock-months, {m['DSCD'].nunique():,} stocks, {m['Country'].nunique()} countries, "
      f"{m['Month'].min()} to {m['Month'].max()}")

countries = pd.read_csv(COUNTRIES_FILE)
countries["country_uc"] = countries["country"].str.upper()
euro_adoption = countries.dropna(subset=["euro_adoption"]).set_index("country_uc")["euro_adoption"].to_dict()
missing_ff = sorted(set(eu.FF_EUROPE_COUNTRIES) - set(m["Country"].unique()))
print("FF Europe countries missing in our data:", missing_ff or "none")

In [ ]:
ub = eu.universe_by_country(m)
n_stocks = ub.pivot(index="Month", columns="Country", values="n_stocks")
me_share = ub.pivot(index="Month", columns="Country", values="me_eur")
me_share = me_share.div(me_share.sum(axis=1), axis=0)

# snapshot: every 5 years, number of stocks and share of European market cap
snap = [p for p in n_stocks.index if p.month == 12 and p.year % 5 == 0] + [n_stocks.index.max()]
display(n_stocks.loc[snap].T.fillna(0).astype(int))
display((100 * me_share.loc[snap].T).round(1))

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
n_stocks.sum(axis=1).plot(ax=ax[0], title="Stocks with a return, all countries")
top = me_share.mean().nlargest(8).index
me_share[top].plot(ax=ax[1], title="Share of EUR market cap (8 largest countries)")
plt.tight_layout()

Look for: sudden jumps in the number of stocks of a country (list or exchange mapping problems), countries
entering late (coverage) and market-cap shares that do not match known market sizes (UK, FR, DE, CH largest).

## 2. Europe-16 vs. Fama-French Europe market

In [ ]:
ff_eu = get_ff_region_factors("Europe", "monthly")
fx = eu.fx_month_end(FX_FILE)                                  # warns if the file holds monthly averages
vw_eur = eu.vw_returns(m, "ret_eur", countries=eu.FF_EUROPE_COUNTRIES).rename("VW_EUR")
vw_usd = eu.eur_to_usd(vw_eur, fx).rename("VW_USD")
print(f"FF Europe {ff_eu.index.min()} to {ff_eu.index.max()}, FX {fx.index.min()} to {fx.index.max()}")

stats_usd, res_usd = ev.compare_to_market(vw_usd, ff_eu, periods_per_year=12, hac_lags=HAC_LAGS)
pd.Series(stats_usd, name="VW Europe-16 (USD)").to_frame()

### 2b. Currency of the FF Europe returns

If FF Europe is in USD, our USD series correlates clearly more with `Mkt-RF + RF` than our EUR series (EUR/USD
volatility is about 8-10% p.a., so the EUR version should have a visibly lower correlation and a tracking
error of roughly that size).

In [ ]:
ff_mkt = (ff_eu["Mkt-RF"] + ff_eu["RF"]).rename("FF_Mkt")
cur = pd.concat([vw_usd, vw_eur, ff_mkt], axis=1, join="inner").dropna()
pd.DataFrame({
    "corr_with_FF": [cur["VW_USD"].corr(cur["FF_Mkt"]), cur["VW_EUR"].corr(cur["FF_Mkt"])],
    "tracking_error_annual": [(cur["VW_USD"] - cur["FF_Mkt"]).std() * np.sqrt(12),
                              (cur["VW_EUR"] - cur["FF_Mkt"]).std() * np.sqrt(12)],
}, index=["our VW in USD", "our VW in EUR"])

In [ ]:
periods = [("1999-2007", "1999-01", "2007-12"), ("2008-2012", "2008-01", "2012-12"),
           ("2013-2019", "2013-01", "2019-12"), ("2020-", "2020-01", str(ff_eu.index.max()))]
display(ev.subperiod_table(vw_usd, ff_eu, periods, periods_per_year=12, hac_lags=HAC_LAGS)
        [["n_obs", "corr", "beta", "alpha_annual", "alpha_t_hac", "tracking_error_annual", "mean_diff_annual"]])

roll = ev.rolling_fit(vw_usd, ff_eu, window=36)
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
roll["corr"].plot(ax=ax[0], title="Rolling 36m correlation with FF Europe")
roll["beta"].plot(ax=ax[1], title="Rolling 36m beta")
ax[1].axhline(1, color="grey", lw=0.8)
plt.tight_layout()

cum = pd.concat([vw_usd, ff_mkt], axis=1, join="inner").dropna()
(1 + cum).cumprod().plot(logy=True, figsize=(15, 4), title="Cumulative return (USD)")
ev.largest_deviations(vw_usd, ff_eu, n=10)

Large single-month deviations point to data problems in a few big stocks (check the stocks with the largest
contribution in that month below) or to differences in the universe (FF excludes some countries' small caps in
early years).

In [ ]:
def contributions(m, month, n=10, countries=eu.FF_EUROPE_COUNTRIES):
    """Largest absolute contributions (weight × return) to the VW return of one month."""
    d = m[(m["Month"] == pd.Period(month, "M")) & m["Country"].isin(countries)].dropna(subset=["ret_eur", "lag_me_eur"])
    d = d.assign(weight=d["lag_me_eur"] / d["lag_me_eur"].sum())
    d = d.assign(contribution=d["weight"] * d["ret_eur"])
    return d.reindex(d["contribution"].abs().sort_values(ascending=False).index)[
        ["DSCD", "Country", "lag_me_eur", "weight", "ret_eur", "ret_local", "contribution"]].head(n)

worst = ev.largest_deviations(vw_usd, ff_eu, n=1).index[0]
print("Month with the largest deviation:", worst)
contributions(m, worst)

## 3. Countries vs. Datastream Total Market indices

Download (Datastream, monthly, from 1992-12-31): `RI~E` of the **"<COUNTRY>-DS Market"** index of every sample
country into one sheet, first column dates (in Navigator: Category *Equity Indices*, search `DS Market`). The
column names as exported (`UK-DS Market`, `SWITZ-DS Market`, `CZECH REP.-DS Market`, ...) are mapped to our country
names by `eu_evaluation.ds_market_country`; new abbreviations go into `DS_MARKET_NAME_ALIASES`.

In [ ]:
vw_country = eu.vw_returns(m, "ret_eur", by="Country")
if TOTMK_FILE.exists():
    idx = eu.index_returns(TOTMK_FILE)
    # columns are named '<COUNTRY>-DS Market' (e.g. 'UK-DS Market'); mapped to our Country names
    idx_by_country = eu.ds_market_by_country(idx, countries=vw_country.columns)
    other = [c for c in idx.columns if eu.ds_market_country(c) not in vw_country.columns]
    if other:
        print("Index columns without a country in our data (ignored):", other)
    print("Countries without index:", sorted(set(vw_country.columns) - set(idx_by_country.columns)) or "none")
    if idx_by_country.empty:
        raise ValueError(f"No '<COUNTRY>-DS Market' column matched. Columns in the file: {list(idx.columns)}")
    cmp = eu.compare_series(vw_country, idx_by_country, {c: c for c in idx_by_country.columns})
    cmp = cmp.join(n_stocks.median().rename("median_n_stocks"))
    display(cmp.sort_values("corr"))
else:
    print(f"{TOTMK_FILE} not found: download the Total Market indices first (section 3 skipped).")

Look for: correlations clearly below 0.95 in large markets, betas far from 1 and large mean differences.
Datastream's total market indices hold a representative sample of the largest stocks (not all), so small caps
explain some tracking error, but not a low correlation.

## 4. Local vs. EUR returns

In [ ]:
lve = eu.local_vs_eur(m, euro_adoption)
display(lve.pivot(index="Country", columns="period", values="share_differs").round(3))

Expected: `after` ≈ 0 for euro countries (legacy-currency lines are fixed multiples after adoption);
`before` and `no euro` ≈ 1 (exchange-rate movements), except for pegged currencies (DKK, BGN, Baltic currencies)
where differences are tiny but still > 1e-6. Non-zero `after` shares point to lines quoted in a foreign currency
(should have been removed by filter 5) or to wrong EUR series.

## 5. Optional: Top 600 vs. STOXX Europe 600 / MSCI Europe

In [ ]:
top600 = eu.top_n_portfolio(m, n=600, countries=[c for c in m["Country"].unique()
                                                 if c not in ("ICELAND",)])
if STOXX_FILE.exists():
    bench = eu.index_returns(STOXX_FILE)
    display(eu.compare_series(top600.to_frame(), bench, {"Top600": bench.columns[0]}))
else:
    print(f"{STOXX_FILE} not found (optional). Top 600 vs. all-country VW for reference:")
    vw_all = eu.vw_returns(m, "ret_eur")
    print(f"corr = {pd.concat([top600, vw_all], axis=1).dropna().corr().iloc[0, 1]:.4f}")

STOXX 600 weights by free-float market cap; ours by total market cap, so a correlation around 0.99 with a small
tracking error is the realistic target.